# Baseline: YOLOv8n (50 epoch)

Pipeline:
1. Làm sạch dữ liệu
2. Chuyển đổi COCO sang YOLO
3. Chia tập dữ liệu
4. Huấn luyện (không cải tiến)

In [1]:
!git clone https://github.com/Shiba-dotcom/waste-detection_project.git

Cloning into 'waste-detection_project'...
remote: Enumerating objects: 3590, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 3590 (delta 5), reused 5 (delta 5), pack-reused 3573 (from 3)
Receiving objects: 100% (3590/3590), 2.58 GiB | 58.04 MiB/s, done.
Resolving deltas: 100% (172/172), done.


In [ ]:
!pip install gdown
# Tải file từ Drive về Kaggle
!gdown --id "17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM" -O raw.zip

# Chỉ cần tạo đến thư mục data (thư mục raw sẽ tự sinh ra khi giải nén)
!mkdir -p /kaggle/working/waste-detection_project/data

# Giải nén vào thư mục data
!unzip -q raw.zip -d /kaggle/working/waste-detection_project/data/

# Dọn dẹp file zip để tránh bị đầy bộ nhớ (Disk quota exceeded)
!rm raw.zip
!rm -rf /kaggle/working/waste-detection_project/.git

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM
From (redirected): https://drive.google.com/uc?id=17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM&confirm=t&uuid=8191af3b-4673-4360-a5b9-65d2f6147b76
To: /kaggle/working/raw.zip
100%|███████████████████████████████████████| 2.62G/2.62G [00:25<00:00, 104MB/s]


In [3]:
%cd /kaggle/working/waste-detection_project
!pip install -r requirements.txt

/kaggle/working/waste-detection_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.7 MB/s eta 0:00:00


In [4]:
# Di chuyển vào đúng thư mục của script rồi mới chạy
%cd /kaggle/working/waste-detection_project/src/data_prep
!python data_cleaning.py

%cd /kaggle/working/waste-detection_project/src
!python Training_dataYolo.py

%cd /kaggle/working/waste-detection_project/src/data_prep
!python split_dataset.py

/kaggle/working/waste-detection_project/src/data_prep
  DATA CLEANING - TACO Dataset

[Load] annotations.json ...
  So anh goc         : 1500
  So annotations goc : 4784
  So categories      : 60

────────────────────────────────────────────────────────────
[Buoc 1] Kiem tra dong trung lap (Duplicates)
────────────────────────────────────────────────────────────
  Duplicate annotations: 0
  Duplicate image IDs: 0
  Duplicate file names: 0

────────────────────────────────────────────────────────────
[Buoc 2] Kiem tra gia tri thieu (Missing Values)
────────────────────────────────────────────────────────────
  Images: Tat ca truong bat buoc day du
  Annotations: Tat ca truong bat buoc day du
  Anh khong co annotation: 0

────────────────────────────────────────────────────────────
[Buoc 3] Kiem tra nhan khong hop le
────────────────────────────────────────────────────────────
  Annotations voi category_id khong hop le: 0
  Categories khong co trong mapping.csv: 1
    - 'Rope & strings' 

In [ ]:
import os, time, glob
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = Path.cwd().parent.parent
YAML_PATH = BASE_DIR / "data/processed/dataset.yaml"
RESULTS = BASE_DIR / "results"
RESULTS.mkdir(exist_ok=True)

print("="*55)
print("  BASELINE: YOLOv8n - 50 Epoch")
print("="*55)

model_path = BASE_DIR / "models/yolov8n.pt"
model = YOLO(str(model_path))
# Train YOLOv8n, 50 epochs, no augmentations
train_results = model.train(
    data     = str(YAML_PATH),
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    name     = "baseline_yolov8n",
    project  = str(RESULTS / "runs"),
    exist_ok = True,
    verbose  = True,
)
# Evaluation
print("\n[2] Danh gia tren tap Test ...")
metrics = model.val(split="test", verbose=False)

map50 = float(metrics.box.map50)
prec = float(metrics.box.mp)
recall = float(metrics.box.mr)
map5095 = float(metrics.box.map)

print(f"\nmAP@0.5: {map50:.4f}")

val_imgs = glob.glob(str(BASE_DIR / "data/processed/images/test/**/*.*"), recursive=True)[:50]
if val_imgs:
    t0 = time.perf_counter()
    for img_path in val_imgs:
        model.predict(img_path, verbose=False)
    elapsed = (time.perf_counter() - t0) / len(val_imgs) * 1000
    print(f"Inference time (avg): {elapsed:.1f} ms/anh")
else:
    elapsed = None

result_row = {
    "Model": "YOLOv8n (baseline, 50 epoch)",
    "mAP@0.5 (%)": round(map50*100, 2),
    "Precision (%)": round(prec*100, 2),
    "Recall (%)": round(recall*100, 2),
    "Inference (ms)": round(elapsed, 1) if elapsed else "N/A",
}
df = pd.DataFrame([result_row])
df.to_csv(RESULTS / "ket_qua_baseline.csv", index=False)